# Price Optimization

Price and quantity are two fundamental measures that determine the bottom line of every business, and seting the right price is one of the most important decisions a company can make. Under-pricing hurts the company’s revenue if consumers are willing to pay more and, on the other hand, over-pricing can hurt in a similar fashion if consumers are less inclined to buy the product at a higher price.

Given this tricky relationship between price and sales, it is necessary to find the optimum price that will maximize product sales and earn most profit.

In [2]:
# Uncomment the line of code below if import doesn't work.
# !pip install statsmodels or %pip install statsmodel
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.formula.api import ols
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
%matplotlib inline

In [5]:
# Reading in the demand data (Currently it is Kaggle data). We are parsing date column as datetime object'
demand_df = pd.read_csv('/content/Retail_Data_SP23.csv', parse_dates = ['date']) # You would have to put in your specific file path in the inverted commas for this line to work
demand_df.head()

,product_id,date,sales,Price
0,P0125,2017-01-02,0.0,4.672649
1,P0125,2017-01-02,0.0,2.158928
2,P0125,2017-01-02,0.0,5.369431
3,P0125,2017-01-02,0.0,4.668526
4,P0125,2017-01-02,0.0,4.576855


In [6]:
# Extracting week from the date column and keeping it as a separate feature(column)

demand_df['week'] = demand_df['date'].dt.isocalendar().week
demand_df.head()

,product_id,date,sales,Price,week
0,P0125,2017-01-02,0.0,4.672649,1
1,P0125,2017-01-02,0.0,2.158928,1
2,P0125,2017-01-02,0.0,5.369431,1
3,P0125,2017-01-02,0.0,4.668526,1
4,P0125,2017-01-02,0.0,4.576855,1


In [7]:
# group the values by meal ID and week to get weekly demand estimates of the number of orders
demand = demand_df.groupby(['week'])['sales'].sum()

# Group the values by meal ID and week to get the checkout price (median price is considered)
price = demand_df.groupby(['week'])['Price'].median()

# # Combine the two dataframes into one single dataframe. By default the index will meal ID and week when groupby is used.
# # reset_index resets the index of the dataframe and creates a variable for the old index. T is for transpose.
price_opt_df = pd.DataFrame([demand, price]).T.reset_index()

# Take a look at the first 5 records(rows) to make sure it is the format we want
price_opt_df.head() 

,week,sales,Price
0,1,1513.0,4.987301
1,2,1634.0,5.099215
2,3,1747.0,5.044154
3,4,1750.0,5.042229
4,5,1680.0,4.963700


In [8]:
# fit OLS model with number of orders as the target variable and checkout price as the independent variable
model_ols = ols("sales ~ Price", data = price_opt_df).fit()# print model summary

# Print the summary of the model
print(model_ols.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.033
Model:                            OLS   Adj. R-squared:                  0.013
Method:                 Least Squares   F-statistic:                     1.693
Date:                Tue, 14 Mar 2023   Prob (F-statistic):              0.199
Time:                        23:45:51   Log-Likelihood:                -384.53
No. Observations:                  52   AIC:                             773.1
Df Residuals:                      50   BIC:                             777.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   8085.2683   4975.351      1.625      0.1

In [9]:
# Look at the demand vs price plot of the new dataframe
px.scatter(price_opt_df, x = 'Price', y = 'sales', trendline = 'ols', trendline_color_override = "red")

# The Relationship Between Revenue and Profit

Revenue and profit are two sides of the same coin, and tell two stories of how a business operates. Revenue tells the tale of sales; profit tells a story about efficiency. Here’s a look at what they mean when combined, based on performance:

## Low Revenue, Low Profit

Low figures on both sides represent a company that’s in dire straits. It’s not bringing in enough money (revenue) and the sales it is making are either at a loss or at a margin that’s not enough to sustain the company (profit). Companies in this position need a dire overhaul of business model and strategy.  

## High Revenue, Low Profit

This combination tells the story of a company that has no problem selling, but does so at poor margins or operates the company in such a way that it’s not retaining enough. There’s optimism for these companies. If they can continue to sell with great success, management can explore opportunities to increase margins or keep costs low to eventually generate better profits. 

## Low Revenue, High Profit

These companies are efficient because they’re able to generate strong profits off of fewer sales. This tends to be the case with big-ticket products or high-end services. Companies realize low sales potential, so they sell at higher margins to sustain the business. it’s a viable model, so long as these companies meet their break-even point with the few sales they do record. 

## High Revenue, High Profit

This is the best-case scenario and one every company strives for. High revenues mean the company has no trouble selling. High profit means all that selling is generating great margins and the company is operating lean enough to retain a majority of its income. The biggest challenge these companies face is tempering expectations in the event of a plateau. 

Ultimately, profit is the most important figure. Companies with high profit have proven their ability to operate sustainably and generate shareholder value. Those without profit aren’t doing what it takes to stay afloat. Meanwhile, revenue figures tend to inform the success (or failure) of a company to maintain profitability. High revenue means plenty of sales; low revenue demands a ramp up in selling. 

In [32]:
# A function that takes in the price, and cost as variables. Price should be a list and cost can either be a list or
# a variable depending on the scenarios you want to run. The output of the function is a plot with revenue and profit
# The default parameters are set if they are not specified it will run with default parameter.
# It is important to pass the argument for model if the name of the model is changed.

def price_revenue(price, cost,  unit_variable_cost_type = "single", model = model_ols):
    
    
    # Check if the type of cost is single or multiple. 
    if unit_variable_cost_type == "single":
        # Create empty list called revenue and profit to store the value
        Revenue = []
        profit = []
        # Loop over the price list and obtain the estimate for quantity based on the ols model fit earlier.
        # Append the values of revenue for each price to the empty list. 
        for i in price:
            quantity = model.params[0] + model.params[1] * i
            Revenue.append(i * quantity)
            profit.append((i-cost) * quantity)
        # Create a dataframe with the values of price revenue and profit as the columns.
        rev = pd.DataFrame({"Price" : price, "Revenue" : Revenue, "Profit": profit})
        
        # Initialize plotly graph objects 
        fig = go.Figure() # Creates an empty plot
        fig.add_trace(go.Scatter(x=price, y=Revenue,
                            mode='lines',
                            name='Estimated Revenue'))  # Plots the revenue value and profit value
        fig.add_trace(go.Scatter(x=price, y=profit,
                            mode='lines+markers',
                            name='Estimated profit'))
        fig.update_traces(mode="markers+lines", hovertemplate=None)
        fig.update_layout(hovermode="x unified", title = "Cost : "  + '$' + str(cost), xaxis_title= "Price")
        return fig.show()

    
    # This condition checks it the type of cost specified is variable
    elif unit_variable_cost_type == "multiple":
        # Create an empty dataframe
        # Loop over each value in the cost list.
        for j in cost:
            # Loop over eeach value in the price list for the specified cost and calculate the quantity based on
            # OLs model built earlier
            cost_inc = pd.DataFrame()
            for i in price:
                quantity = model.params[0] + model.params[1] * i
                Revenue = (i * quantity)
                profit = (i-j) * quantity
                # Store the value of Revenue, and profit for the associated cost and price.
                rev = pd.DataFrame({"Cost": [j], "Price" : [i], "Revenue" : [Revenue], "Profit" : [profit], 
                                   "Quantity" : quantity})
                # Append the values to the empty dataframe
                cost_inc = cost_inc.append(rev)

                
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=cost_inc['Price'], y=cost_inc['Revenue'],
                                mode='lines',
                                name='Estimated Revenue'))
            fig.add_trace(go.Scatter(x=cost_inc['Price'], y=cost_inc['Profit'],
                                mode='lines+markers',
                                name='Estimated profit'))
            fig.update_traces(mode="markers+lines", hovertemplate=None)
            fig.update_layout(hovermode="x unified", title = "Cost :"  + '$' + str(j), xaxis_title= "Price")
            fig.show()

In [33]:
#A range of Price to find the optimum pricing
price = list(np.arange(1, 4, 0.1)) #This value should be updated based on the survey [100, 250, 150]

# Assuming variable Cost
cost = [0.9, 1, 2]

# You would need to call this function to obtain the revenue or profit plot
price_revenue(price = price, cost = cost, unit_variable_cost_type = "multiple", model = model_ols)


<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Us

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Us

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.

<ipython-input-32-191b6d2c8ce3>:52: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Us